# Reinforcement Learning Statistics Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Reinforcement Learning Statistics**.
It demonstrates the statistical foundations of RL: estimation, exploration, and policy evaluation.

Topics covered:

1. Multi-armed bandit simulation
2. Epsilon-greedy exploration
3. Upper Confidence Bound (UCB) action selection
4. Temporal Difference (TD) learning
5. Q-learning on a toy environment
6. Off-policy evaluation
7. Reward uncertainty intervals
8. Summary table
9. Mini exercises

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. Multi-Armed Bandit Setup

A multi-armed bandit has K arms with unknown reward distributions.
The goal is to maximize total reward through a balance of exploration and exploitation.

Here we set up three arms with true Gaussian reward distributions.

In [ ]:
true_means = np.array([0.2, 0.5, 0.8])
true_stds  = np.array([0.1, 0.1, 0.1])
n_arms = len(true_means)

pd.DataFrame({
    'Arm':       [1, 2, 3],
    'True mean': true_means,
    'True std':  true_stds
})

## 2. Epsilon-Greedy Exploration

With probability \epsilon, choose a random arm (explore).
Otherwise, choose the arm with the highest estimated mean (exploit).

Value update (incremental mean):

$$Q_{t+1}(a) = Q_t(a) + \frac{1}{N_t(a)}\left[R_t - Q_t(a)\right]$$

In [ ]:
eps = 0.1
Q   = np.zeros(n_arms)
N   = np.zeros(n_arms)
chosen_rewards = []

for t in range(300):
    a = np.random.randint(n_arms) if np.random.rand() < eps else np.argmax(Q)
    r = np.random.normal(true_means[a], true_stds[a])
    N[a] += 1
    Q[a] += (r - Q[a]) / N[a]
    chosen_rewards.append(r)

pd.DataFrame({'Arm': [1, 2, 3], 'Estimated mean': Q.round(4), 'Pull count': N.astype(int)})

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.cumsum(chosen_rewards) / np.arange(1, len(chosen_rewards) + 1))
plt.axhline(true_means.max(), linestyle='--', label='Best arm mean')
plt.title('Running Average Reward (Epsilon-Greedy, eps=0.1)')
plt.xlabel('Step')
plt.ylabel('Average reward')
plt.legend()
plt.show()

## 3. Upper Confidence Bound (UCB) Action Selection

UCB selects the arm with the highest optimistic estimate:

$$a^* = \arg\max_a \left[Q(a) + c\sqrt{\frac{\ln t}{N(a)}}\right]$$

The confidence bonus ensures under-explored arms get tried.

In [ ]:
Q_ucb   = np.zeros(n_arms)
N_ucb   = np.ones(n_arms)   # initialize to 1 to avoid log(0)
ucb_rewards = []
c = 2

for t in range(1, 301):
    ucb_vals = Q_ucb + c * np.sqrt(np.log(t + 1) / N_ucb)
    a = np.argmax(ucb_vals)
    r = np.random.normal(true_means[a], true_stds[a])
    N_ucb[a] += 1
    Q_ucb[a] += (r - Q_ucb[a]) / N_ucb[a]
    ucb_rewards.append(r)

pd.DataFrame({'Arm': [1, 2, 3], 'UCB mean estimate': Q_ucb.round(4), 'Pull count': N_ucb.astype(int)})

## 4. Temporal Difference (TD) Learning

TD learning estimates state values from bootstrapped updates:

$$V(s) \leftarrow V(s) + \alpha \left[r + \gamma V(s') - V(s)\right]$$

where:
- \alpha = learning rate
- \gamma = discount factor
- r + \gamma V(s') = TD target

In [ ]:
V      = np.zeros(5)    # 5 states: 0,1,2,3,4
alpha  = 0.1
gamma  = 0.9

for _ in range(400):
    s     = np.random.randint(0, 4)
    r     = 1 if s == 3 else 0    # reward only at state 3
    s_next = min(s + 1, 4)
    V[s]  += alpha * (r + gamma * V[s_next] - V[s])

pd.DataFrame({'State': np.arange(5), 'Value estimate': V.round(4)})

## 5. Q-Learning on a Toy Environment

Q-learning estimates action-value functions via off-policy updates:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\right]$$

In [ ]:
n_states  = 4
n_actions = 2
Q_table   = np.zeros((n_states, n_actions))

for _ in range(800):
    s = np.random.randint(0, n_states - 1)
    a = np.random.randint(n_actions)
    r = 1 if (s == 2 and a == 1) else 0
    s_next = min(s + 1, n_states - 1)
    Q_table[s, a] += 0.1 * (r + 0.9 * np.max(Q_table[s_next]) - Q_table[s, a])

q_df = pd.DataFrame(Q_table, columns=['Action 0', 'Action 1'])
q_df.index.name = 'State'
q_df.round(4)

## 6. Off-Policy Evaluation

Off-policy evaluation estimates how a **target policy** \pi_t would perform using data
collected under a different **behavior policy** \pi_b.

Importance-weighted estimator:

$$\hat{V}(\pi_t) = \frac{1}{n}\sum_i \frac{\pi_t(a_i)}{\pi_b(a_i)} R_i$$

In [ ]:
n_ope = 400
behavior_prob = 0.5    # behavior policy: 50/50
target_prob   = 0.8    # target policy: mostly action 1

actions_ope = np.random.binomial(1, behavior_prob, n_ope)
rewards_ope = np.random.normal(0.5 + 0.4 * actions_ope, 0.1)

weights_ope = np.where(actions_ope == 1,
                        target_prob / behavior_prob,
                        (1 - target_prob) / (1 - behavior_prob))

naive_est       = rewards_ope.mean()
off_policy_est  = np.mean(weights_ope * rewards_ope)
true_target_val = 0.5 + 0.4 * target_prob   # analytical expected reward under target

pd.DataFrame({
    'Metric':  ['Naive (behavior policy) estimate', 'Off-policy (IW) estimate', 'True target value (analytical)'],
    'Value':   [naive_est, off_policy_est, true_target_val]
})

## 7. Reward Uncertainty Intervals

Confidence intervals on accumulated reward help assess policy stability.

$$\bar{R} \pm 1.96 \frac{s_R}{\sqrt{T}}$$

In [ ]:
r_arr   = np.array(chosen_rewards)
mean_r  = r_arr.mean()
std_r   = r_arr.std(ddof=1)
ci_low  = mean_r - 1.96 * std_r / np.sqrt(len(r_arr))
ci_high = mean_r + 1.96 * std_r / np.sqrt(len(r_arr))

pd.DataFrame({
    'Metric':  ['Mean reward', '95% CI low', '95% CI high', 'Regret (best - actual)'],
    'Value':   [mean_r, ci_low, ci_high, true_means.max() - mean_r]
})

## 8. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Measure':  [
        'EG final running average reward',
        'UCB final running average reward',
        'Best TD value estimate (state 4)',
        'Best Q-value',
        'Off-policy estimate',
        'Reward 95% CI width'
    ],
    'Value': [
        round(np.mean(chosen_rewards), 4),
        round(np.mean(ucb_rewards), 4),
        round(V[4], 4),
        round(np.max(Q_table), 4),
        round(off_policy_est, 4),
        round(ci_high - ci_low, 4)
    ]
})
summary

## 9. Mini Exercises

Try these on your own:

1. Change the epsilon value to 0.01 (greedy) and 0.5 (exploratory) and compare cumulative regret over 500 steps.
2. Add a fourth arm with mean 0.65 and observe how UCB allocates pulls compared to epsilon-greedy.
3. Change the discount factor gamma from 0.9 to 0.5 in TD learning and compare value estimates.
4. In Q-learning, add a reward at two states instead of one and visualize the learned Q-table.
5. Change the behavior policy to 0.3 in the off-policy cell and verify the IW estimate still recovers the target value.
6. Compare epsilon-greedy and UCB cumulative reward on the same plot and measure which converges faster.
7. Simulate a non-stationary bandit where the true arm means shift halfway through and observe how the algorithms adapt.
8. Apply the reward confidence interval to a longer run (1000 steps) and compare the CI width.

These exercises are especially useful for AI decision systems, adaptive experimentation, recommendation engines, and digital twin control.